# Agent的高级用法-错误处理机制

## 情况1：设置为True/False/固定字符串

举例1：handle_errors 设置为True

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

In [2]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=True
        #handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)
# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='6f529e12-12c3-4136-b1b4-418383aaa234'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 416,
                    'total_tokens': 504,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 416}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-8d434404-6a98-9fdb-add4-5930978c88fd',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bb8-8eab-7092-8881-0a88749c075d-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_ac9bee78f2f64b41a4a477f6',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_317b4662d6784de9a80bf63f',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 416,
                'output_tokens': 88,
                'total_tokens': 504,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='d7e54fb4-816d-49cb-a616-bf1059203c90',
            tool_call_id='call_ac9bee78f2f64b41a4a477f6'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='27a04301-b7af-47b7-9ea4-d0764c726cf8',
            tool_call_id='call_317b4662d6784de9a80bf63f'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 40,
                    'prompt_tokens': 580,
                    'total_tokens': 620,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 580}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-48bd827f-2f9e-95c4-ba2c-d026057de093',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bb8-95ef-7913-8e54-6cbdb2ab8706-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_fac1ccfedf944355a253f1b6',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 580,
                'output_tokens': 40,
                'total_tokens': 620,
                'input_token_details': {'cache_read': 0},
 

举例2：handle_errors 设置为False

In [3]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=False
        #handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)
# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

举例3：handle_errors 设置为固定的字符串

In [4]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors="请检查输入的数据"
        #handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)
# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='e832303e-bfe4-4ff6-a888-e6e84a0da455'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 416,
                    'total_tokens': 504,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 416}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-413b5f86-7022-939b-a9e6-64e23e0467c2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bbf-592d-7ab0-8bae-904418095bd3-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_4810ccabfa274f47bdcd8bce',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_dc418160915c4669ba1c8eac',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 416,
                'output_tokens': 88,
                'total_tokens': 504,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='请检查输入的数据',
            name='ContactInfo',
            id='af11f63f-812c-45fb-b786-dfde2037a867',
            tool_call_id='call_4810ccabfa274f47bdcd8bce'
        ),
        ToolMessage(
            content='请检查输入的数据',
            name='EventDetails',
            id='58b96432-57d1-4d0e-943c-aecc3b8d6ba5',
            tool_call_id='call_dc418160915c4669ba1c8eac'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 44,
                    'prompt_tokens': 534,
                    'total_tokens': 578,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 534}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-43ec79bd-3eb8-987b-9cbe-eb47fa4e4845',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bbf-5fdd-7812-bb58-9fdc8058d1ae-0',
            tool_calls=[
                {
                    'name': 'EventDetails',
                    'args': {'date': '2026-07-15', 'event_name': '公司年会'},
                    'id': 'call_2d7a159512bd4fdab23a23de',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 534,
                'output_tokens': 44,
                'total_tokens': 578,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='提取完成！',
            name='EventDetails',
            id='6aff78ea-0a26-49cf-999c-25e3084a4c4f',
            tool_call_id='call_2d7a159512bd4fdab23a23de'
        )
    ],

## 情况2：设置为指定异常类型

举例1：

In [7]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=(
            MultipleStructuredOutputsError,
            StructuredOutputValidationError
        )
        # handle_errors=(
        #     StructuredOutputValidationError
        # )
        #handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)
# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='3ce37423-cfe3-489b-a0ec-41e9974031fe'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 416,
                    'total_tokens': 504,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 416}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-b61abd39-06db-9ee7-a756-55111fe6cbbd',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bc6-21ca-78c1-a301-484fb4fe7d18-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_3edbf16b1de3402b8126bfb7',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_9ded0890ab82440f8a325b5e',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 416,
                'output_tokens': 88,
                'total_tokens': 504,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='4028787b-3dc4-4180-b15e-0802ca950a44',
            tool_call_id='call_3edbf16b1de3402b8126bfb7'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='df743dad-1779-4470-bc82-d6c3627878ff',
            tool_call_id='call_9ded0890ab82440f8a325b5e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 40,
                    'prompt_tokens': 580,
                    'total_tokens': 620,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 580}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-53ca77b6-b691-95ec-9ac1-e2d8b367b155',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bc6-2952-7b71-b262-249b91777097-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_2de125c1598d4e5596513c4e',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 580,
                'output_tokens': 40,
                'total_tokens': 620,
                'input_token_details': {'cache_read': 0},
 

## 情况3：设置为自定义错误处理函数

指定异常处理函数并返回字符串时，LangChain会在遇到异常时自动重试并将异常处理函数的返回值作为ToolMessage的内容。

我们也可以选择在异常处理函数中抛出异常。

举例：


In [9]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

# 自定义错误处理的函数
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理器"""

    error_str = str(error)

    print(f"捕获到的错误类型是：{type(error).__name__}")
    print(f"错误详情：{error_str}")

    if isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回"
    elif isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求"
    else:
        return f"Error:{error_str}"


class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=custom_error_handler
        #handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)
# for msg in result["messages"]:
#     msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

捕获到的错误类型是：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.


{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='bf1e194d-431a-465e-ac93-bee0046e7671'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 416,
                    'total_tokens': 504,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 416}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-798e4cb5-2c60-9a02-ae5b-1b0d2381b398',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bcd-9a5f-77f1-a25c-1e52d7c7401b-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_37a2aa8c8f1648949c4b5169',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_2ba49fe3c3244f83ae77a4f3',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 416,
                'output_tokens': 88,
                'total_tokens': 504,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回',
            name='ContactInfo',
            id='2260bd21-6f55-4229-aa05-71977c5b339d',
            tool_call_id='call_37a2aa8c8f1648949c4b5169'
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回',
            name='EventDetails',
            id='8e49afab-e8e3-42dc-9ecc-14ee4d45417b',
            tool_call_id='call_2ba49fe3c3244f83ae77a4f3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 40,
                    'prompt_tokens': 546,
                    'total_tokens': 586,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 546}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-93922227-ec6f-92e9-913e-13f1588fce4e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bcd-a242-7221-a2b0-6feec2ce9fa1-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_628693b01bbe4dcd81cc0c98',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 546,
                'output_tokens': 40,
                'total_tokens': 586,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='提取完成！',
            name='ContactInfo',
            id='03ba2529-2b06-4d2a-8479-32ed76389926',
            tool_call_id='call_628693b01bbe4dcd8